In [ ]:
import cv2
import numpy as np
import json
import datetime
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import Video, display, HTML
from ipywidgets import widgets
import io

# Initialize YOLOv8 model
model = YOLO("yolov8n.pt")  

# Cricket object classes mapping
CRICKET_CLASSES = {
    'ball': 32,       # sports ball
    'bat': 39,        # baseball bat
    'player': 0,      # person
    'stumps': 9       # bench (placeholder)
}

def estimate_3d_position(x2d, y2d, frame_width, frame_height, object_class):
    """Improved 3D position estimation"""
    x_norm = x2d / frame_width
    y_norm = y2d / frame_height
    
    # Different depth assumptions for different objects
    if object_class == 'ball':
        z = 0.5 + (1 - y_norm) * 2  # Higher when ball is higher in frame
        x = (x_norm - 0.5) * 3
        y = (0.5 - y_norm) * 3
    elif object_class == 'bat':
        z = 0.3 + (1 - y_norm) * 0.7
        x = (x_norm - 0.5) * 2
        y = (0.5 - y_norm) * 1.5
    else:  # player and stumps
        z = 0.1 + (1 - y_norm) * 0.4
        x = (x_norm - 0.5) * 2
        y = 0  # Grounded objects
        
    return round(x, 2), round(y, 2), round(z, 2)

# Create upload widget
upload = widgets.FileUpload(
    accept='.mp4,.avi,.mov',
    multiple=False,
    description='Upload Video'
)

# Create output widgets
output = widgets.Output()
progress = widgets.FloatProgress(value=0, min=0, max=100, description='Processing:')

# Display widgets
display(upload)
display(progress)
display(output)

def process_upload(change):
    if not upload.value:
        return
    
    with output:
        output.clear_output()
        print("Starting video processing...")
        
        # Get uploaded video
        video_name = next(iter(upload.value))
        video_bytes = upload.value[video_name]['content']
        
        # Write to temporary file
        with open('temporary_video.mp4', 'wb') as f:
            f.write(video_bytes)
            
        # Process video
        cap = cv2.VideoCapture('temporary_video.mp4')
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        ball_trajectory = []
        bat_positions = []
        batsman_positions = []
        stump_positions = []
        
        frame_count = 0
        progress.max = total_frames
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            progress.value = frame_count
            
            # Process every 5th frame for speed
            if frame_count % 5 == 0:
                results = model(frame)[0]
                
                for box in results.boxes:
                    x1, y1, x2, y2 = map(float, box.xyxy[0])
                    cls = int(box.cls)
                    conf = float(box.conf)
                    xc, yc = (x1 + x2) / 2, (y1 + y2) / 2
                    
                    obj_class = None
                    if cls == CRICKET_CLASSES['ball'] and conf > 0.3:
                        obj_class = 'ball'
                    elif cls == CRICKET_CLASSES['bat'] and conf > 0.3:
                        obj_class = 'bat'
                    elif cls == CRICKET_CLASSES['player'] and conf > 0.5:
                        obj_class = 'player'
                    elif cls == CRICKET_CLASSES['stumps'] and conf > 0.4:
                        obj_class = 'stumps'
                        
                    if obj_class:
                        x3d, y3d, z3d = estimate_3d_position(xc, yc, frame_width, frame_height, obj_class)
                        timestamp = round(frame_count / fps, 2)
                        
                        if obj_class == 'ball':
                            ball_trajectory.append({"x": x3d, "y": y3d, "z": z3d, "t": timestamp})
                        elif obj_class == 'bat':
                            bat_positions.append({"x": x3d, "y": y3d, "z": z3d, "t": timestamp})
                        elif obj_class == 'player':
                            batsman_positions.append({"x": x3d, "y": y3d, "z": z3d, "t": timestamp})
                        elif obj_class == 'stumps':
                            stump_positions.append({"x": x3d, "y": y3d, "z": z3d, "t": timestamp})
            
            frame_count += 1
            if frame_count > 150:  # Limit for demo
                break
                
        cap.release()
        
        # Prepare results
        batsman_leg_position = batsman_positions[-1] if batsman_positions else {"x": 0, "y": 0, "z": 0, "t": 0}
        bat_position = bat_positions[-1] if bat_positions else {"x": 0, "y": 0, "z": 0, "t": 0}
        
        # Find unique stumps (simple approach)
        unique_stumps = []
        if stump_positions:
            stump_df = pd.DataFrame(stump_positions)
            unique_stumps = stump_df.groupby(['x', 'y', 'z']).mean().reset_index().to_dict('records')[:3]
        
        # Create output
        output_data = {
            "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
            "video_info": {
                "filename": video_name,
                "duration": round(frame_count / fps, 2),
                "frames_processed": frame_count
            },
            "ball_trajectory": ball_trajectory,
            "bat_position": bat_position,
            "batsman_leg_position": batsman_leg_position,
            "stump_coordinates": unique_stumps
        }
        
        print("\nProcessing complete! Results:")
        print(json.dumps(output_data, indent=2))
        
        # Visualization
        if ball_trajectory:
            df = pd.DataFrame(ball_trajectory)
            fig = plt.figure(figsize=(10, 6))
            ax = fig.add_subplot(111, projection='3d')
            
            ax.scatter(df['x'], df['y'], df['z'], c=df['t'], cmap='viridis')
            ax.set_xlabel('X Position (meters)')
            ax.set_ylabel('Y Position (meters)')
            ax.set_zlabel('Z Position (meters)')
            ax.set_title('3D Ball Trajectory')
            plt.show()
        
        # Show processed video
        display(Video('temporary_video.mp4', embed=True, width=600))

# Observe upload changes
upload.observe(process_upload, names='value')

: 